# Context Managers in PythonContext managers provide a clean way to manage **resources** (files, connections, locks) by ensuring proper setup and teardown.### The `with` Statement```pythonwith expression as variable:    # code block```The context manager guarantees:- **Setup** happens before the block (enter)- **Cleanup** happens after the block, even if an exception occurs (exit)> **Prerequisite**: Exception handling, classes, generators.

## The Context Manager ProtocolAny object that implements `__enter__` and `__exit__` can be used as a context manager.| Method | Timing | Parameters ||--------|--------|------------|| `__enter__(self)` | Before the `with` block | None || `__exit__(self, exc_type, exc_val, exc_tb)` | After the `with` block | Exception info (or `None, None, None`) |If `__exit__` returns `True`, the exception is **suppressed**.

In [ ]:
class FileManager:    """Context manager for file operations."""    def __init__(self, filename, mode='r'):        self.filename = filename        self.mode = mode        self.file = None    def __enter__(self):        print(f"Opening {self.filename}...")        self.file = open(self.filename, self.mode)        return self.file    def __exit__(self, exc_type, exc_val, exc_tb):        print(f"Closing {self.filename}...")        if self.file:            self.file.close()        return Falsewith FileManager("cm_sample.txt", "w") as f:    f.write('Hello from custom context manager!\n')print("File written successfully!")with FileManager("cm_sample.txt", "r") as f:    content = f.read()print(f"Content: {content}")

## Context Manager for Exception Handling

In [ ]:
class SuppressError:    """Context manager that suppresses specified exceptions."""    def __init__(self, *exceptions):        self.exceptions = exceptions    def __enter__(self):        return self    def __exit__(self, exc_type, exc_val, exc_tb):        if exc_type is not None and issubclass(exc_type, self.exceptions):            print(f"Suppressed {exc_type.__name__}: {exc_val}")            return True        return Falseprint("Trying division by zero:")with SuppressError(ZeroDivisionError, ValueError):    result = 10 / 0    print("This won't be reached")print("Execution continued after suppression!")print("\nTrying ValueError:")with SuppressError(ValueError):    int("not a number")print("ValueError suppressed!")

## The `contextlib` Module| Utility | Description ||---------|-------------|| `@contextmanager` | Turn a generator into a context manager || `closing()` | Call `close()` on exit || `suppress()` | Suppress specified exceptions || `redirect_stdout` | Redirect output streams || `nullcontext()` | Do-nothing context manager |

In [ ]:
from contextlib import contextmanager@contextmanagerdef open_file(filename, mode='r'):    """Context manager for file operations using @contextmanager."""    print(f"  [setup] Opening {filename}")    f = open(filename, mode)    try:        yield f    finally:        print(f"  [cleanup] Closing {filename}")        f.close()with open_file("cm_gen.txt", "w") as f:    f.write('Created with @contextmanager!\n')with open_file("cm_gen.txt", "r") as f:    print(f'  Content: {f.read().strip()}')

In [ ]:
from contextlib import closing, suppress, redirect_stdout, nullcontextimport ioclass Connection:    def close(self):        print("Connection closed")    def send(self, data):        print(f"Sending: {data}")conn = Connection()with closing(conn):    conn.send("Hello")with suppress(FileNotFoundError):    with open("nonexistent.txt") as f:        print(f.read())print("FileNotFoundError was suppressed!")buffer = io.StringIO()with redirect_stdout(buffer):    print("This goes to the buffer")    print("Not to the console")print(f"Captured output: {buffer.getvalue()}")with nullcontext() as ctx:    print(f'nullcontext returns: {ctx}')

## Nested and Chained Context Managers

In [ ]:
with open("source.txt", "w") as src, open("dest.txt", "w") as dst:    src.write("Source content")    dst.write("Destination content")print("Both files written in single with statement")with open("source.txt") as src:    with open("dest.txt") as dst:        content = src.read()        print(f"Read: {content}")from contextlib import ExitStackfilenames = ["file1.txt", "file2.txt", "file3.txt"]with ExitStack() as stack:    files = [stack.enter_context(open(f, "w")) for f in filenames]    for i, f in enumerate(files):        f.write(f"Content of file {i+1}\n")print("All files managed with ExitStack closed automatically!")import osfor f in filenames + ['source.txt', 'dest.txt', 'cm_sample.txt', 'cm_gen.txt']:    if os.path.exists(f):        os.remove(f)print("Cleanup complete.")

## Interview Problems### Problem 1: Timing Context Manager

In [ ]:
import timeclass Timer:    """Context manager that measures execution time."""    def __enter__(self):        self.start = time.perf_counter()        return self    def __exit__(self, exc_type, exc_val, exc_tb):        self.elapsed = time.perf_counter() - self.start        print(f'Elapsed time: {self.elapsed:.6f}s')        return Falsewith Timer() as t:    total = sum(x ** 2 for x in range(100000))    print(f"Sum of squares: {total}")from contextlib import contextmanager@contextmanagerdef timer_context():    start = time.perf_counter()    try:        yield    finally:        elapsed = time.perf_counter() - start        print(f'[timer] {elapsed:.6f}s')with timer_context():    time.sleep(0.2)print("Done")

### Problem 2: Database Connection Context Manager

In [ ]:
class DatabaseConnection:    """Simulated database connection with context manager."""    def __init__(self, host, database):        self.host = host        self.database = database        self.connected = False        self.transaction_active = False    def __enter__(self):        self.connect()        return self    def __exit__(self, exc_type, exc_val, exc_tb):        if exc_type is not None:            print(f'  Exception occurred: {exc_type.__name__}: {exc_val}')            if self.transaction_active:                self.rollback()        self.close()        return False    def connect(self):        print(f'  Connecting to {self.database}@{self.host}...')        self.connected = True        print("  Connected!")    def close(self):        if self.connected:            print(f'  Closing connection to {self.database}...')            self.connected = False    def query(self, sql):        if not self.connected:            raise RuntimeError("Not connected")        print(f'  Executing: {sql}')        return [{"result": "data"}]    def begin_transaction(self):        self.transaction_active = True        print("  Transaction started")    def commit(self):        self.transaction_active = False        print("  Transaction committed")    def rollback(self):        self.transaction_active = False        print("  Transaction rolled back")print("=== Successful transaction === " )with DatabaseConnection("localhost", "mydb") as db:    db.begin_transaction()    db.query("SELECT * FROM users")    db.commit()print("\n=== Transaction with rollback ===")try:    with DatabaseConnection("localhost", "mydb") as db:        db.begin_transaction()        db.query("INSERT INTO users VALUES ('Alice')")        raise ValueError("Something went wrong!")except ValueError:    print("Exception caught outside")

### Problem 3: Temporary Directory Context Manager

In [ ]:
import tempfileimport shutilimport osfrom pathlib import Pathclass TempDirectory:    """Context manager for a temporary directory."""    def __enter__(self):        self.path = Path(tempfile.mkdtemp())        print(f'Created temp directory: {self.path}')        return self.path    def __exit__(self, exc_type, exc_val, exc_tb):        shutil.rmtree(self.path)        print(f'Removed temp directory: {self.path}')        return Falseprint("=== TempDirectory test ===")with TempDirectory() as tmpdir:    test_file = tmpdir / "test.txt"    test_file.write_text("Temporary content")    print(f'  Created: {test_file}')    print(f'  Content: {test_file.read_text()}')    print(f'  Directory exists inside with: {tmpdir.exists()}')print(f'Directory exists after with: {tmpdir.exists()}')from contextlib import contextmanager@contextmanagerdef temp_directory():    path = Path(tempfile.mkdtemp())    try:        yield path    finally:        shutil.rmtree(path)with temp_directory() as tmp:    print(f'  Temp dir: {tmp}')    (tmp / "hello.txt").write_text("Hi!")    print(f'  Files: {list(tmp.iterdir())}')print(f'  Cleaned up: {not tmp.exists()}')

## Summary| Approach | Protocol | Use Case ||----------|----------|----------|| Class-based | `__enter__` / `__exit__` | Complex setup/teardown || `@contextmanager` | `yield` | Simple, generator-based || `closing()` | `close()` | Objects with close method || `suppress()` | Exception suppression | Ignoring specific errors || `ExitStack` | Dynamic contexts | Variable number of managers || `nullcontext()` | No-op | Conditional contexts |**Next**: Move to **04_Regex.ipynb** for regular expressions.